In [1]:
%%capture
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [2]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [23]:
names = open('names.txt').read().splitlines()

vocab = sorted(set(''.join(names) + '.'))
stoi = {ch: i+1 for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(stoi) + 1

In [24]:
def encode(name: str) -> list[int]:
    return [stoi[s] for s in name]

def decode(seq: list[int]) -> str:
    return ''.join([itos[i] for i in seq])

In [22]:
for name in names:
    print(name)
    break

emma


In [25]:
X = []
Y = []

for name in names:
    name = '.' + name + '.'
    seq = torch.tensor(encode(name))

    X.append(seq[:-1])
    Y.append(seq[1:])

In [27]:
from torch.utils.data import Dataset, DataLoader

class NamesDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [28]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = pad_sequence(xs, batch_first=True, padding_value=0)
    ys = pad_sequence(ys, batch_first=False, padding_value=0)

    return xs, ys

In [29]:
train_size = int(0.8 * len(X))
Xtr, Xts = X[:train_size], X[train_size:]
Ytr, Yts = Y[:train_size], Y[train_size:]

Dtr = NamesDataset(Xtr, Ytr)
Dts= NamesDataset(Xts, Yts)

In [30]:
Dltr = DataLoader(Dtr, batch_size=32, shuffle=True, drop_last=True, collate_fn=collate_fn)
Dlts = DataLoader(Dts, batch_size=32, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [31]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, emb_d=32, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_d, padding_idx=0)

        self.rnn = nn.RNN(emb_d, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, h = self.rnn(x)
        logits = self.fc(out)
        return logits

In [32]:
model = CharRNN(vocab_size=vocab_size, emb_d=32, hidden_dim=128).to(device)

In [47]:
criterian = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train_func(model, dataloader, optimizer, criterian, device):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)

        y = y.transpose(0, 1)
        logits = logits.reshape(-1, vocab_size)
        y = y.reshape(-1)
        loss = criterian(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [35]:
for x, y in Dltr:
    break

In [48]:
@torch.no_grad()
def evaluate(model, dataloader, criterian, device):
    model.eval()
    total_loss = 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        logits = model(x)

        y = y.transpose(0, 1)
        logits = logits.reshape(-1, vocab_size)
        y = y.reshape(-1)
        loss = criterian(logits, y)

        total_loss += loss.item()


    return total_loss / len(dataloader)

In [49]:
epochs = 20

for epoch in range(epochs):
    train_loss = train_func(
        model, Dltr, optimizer, criterian, device
    )

    val_loss = evaluate(
        model, Dlts, criterian, device
    )

    print(f"Train loss: {train_loss} | Test loss: {val_loss}")

Train loss: 2.2286327208578585 | Test loss: 2.4168139678328786
Train loss: 2.1591939003765583 | Test loss: 2.4305625934505937
Train loss: 2.1474528597295284 | Test loss: 2.418421649814245
Train loss: 2.1448602426052092 | Test loss: 2.418175124410373
Train loss: 2.148605260550976 | Test loss: 2.4173571909244975
Train loss: 2.152357335984707 | Test loss: 2.42719533253665
Train loss: 2.1535443940758707 | Test loss: 2.4067137526042424
Train loss: 2.159364193081856 | Test loss: 2.4118177309558164
Train loss: 2.155489827543497 | Test loss: 2.4306937556954757
Train loss: 2.1686032155156134 | Test loss: 2.4557705281385735
Train loss: 2.17060839176178 | Test loss: 2.4472313680459017
Train loss: 2.1730557069182397 | Test loss: 2.4410548637162393
Train loss: 2.174354976117611 | Test loss: 2.434401647961555
Train loss: 2.175681074857712 | Test loss: 2.449165396429413
Train loss: 2.180578330606222 | Test loss: 2.413861576004408
Train loss: 2.1788726380467414 | Test loss: 2.443080672577246
Train los